In [1]:
import pandas as pd
import altair as alt

# Seminar: the Long-Run Prices Dataset

A notebook using the LRPD, described [here](https://cep.lse.ac.uk/pubs/download/occasional/op055.pdf).

</br></br></br></br>


Let's take a look at the prices dataset first

In [2]:
prices_df = pd.read_parquet('https://autocpi-public.s3.eu-west-2.amazonaws.com/lrpd/db_prices.parquet')
prices_df.describe()

,quote_date,shop_code,item_id_raw,region,price,item_id
count,4.836896e+07,4.836896e+07,4.836897e+07,4.836896e+07,4.836896e+07,4.836897e+07
mean,2.007762e+05,4.771270e+02,3.880409e+05,6.679112e+00,4.960397e+01,3.883983e+05
std,1.060226e+03,1.531775e+03,1.467557e+05,3.407499e+00,2.058161e+02,1.466723e+05
min,1.988020e+05,1.000000e+00,2.101010e+05,1.000000e+00,1.000000e-02,2.101010e+05
25%,1.998110e+05,3.900000e+01,2.129170e+05,3.000000e+00,1.490000e+00,2.129180e+05
50%,2.008050e+05,8.800000e+01,4.301280e+05,7.000000e+00,4.850000e+00,4.301320e+05
75%,2.017070e+05,8.020000e+02,5.104060e+05,9.000000e+00,1.999000e+01,5.104070e+05
max,2.025100e+05,2.007100e+04,6.404060e+05,1.300000e+01,4.400000e+04,6.404060e+05


We've got 48 million observations and a dates from 1982 to 2025.
</br></br></br></br>


We also need the items data to understand what each product is.

In [3]:
items_df = pd.read_parquet('https://autocpi-public.s3.eu-west-2.amazonaws.com/lrpd/db_item.parquet')
items_df.head()

,item_id,description,date_quote_s,date_quote_e,n_obs
0,210101,LARGE LOAF-WHITE-SLICED-800G,198802,200401,36039
1,210102,LARGE LOAF-WHITE-UNSLICED-800G,198802,202510,56917
2,210105,LARGE WHOLEMEAL LOAF-UNSLICED,198802,200301,27161
3,210106,SIX BREAD ROLLS-WHITE/BROWN,198802,202510,67469
4,210107,"BROWN LOAF,400G,SLICED-GRAN",198903,200401,29361


</br></br></br>
# Simple chart: the price of milk

Let's chart the price of milk. First we need to work out what the `item_id` is. Let's look in the items_df to find out.
</br></br>

In [10]:
items_df[items_df['description'].str.contains('butter', case=False)]

,item_id,description,date_quote_s,date_quote_e,n_obs
127,211301,BUTTER-HOME PRODUCED-250G,198802,201301,47553
128,211302,BUTTER NEW ZEALAND,199601,199601,192
129,211303,BUTTER DANISH,199601,199601,185
130,211304,BUTTER-IMPORTED 250G,199602,201301,28928
131,211305,SPREADABLE BUTTER,201302,202510,35311
132,211306,BUTTER 250G SPEC COO,201302,202510,35655
138,211409,"PEANUT BUTTER, JAR, 225-350G",201902,202510,18676


Let's go for `211710` - `MILK SEMI-PER 2 PINTS/1.136 L`. It's got a long timeseries (1992-202510) and is probably representative.

</br></br></br></br>

First, let's just plot the mean, median and median price of milk over time.

In [16]:
butter = prices_df.query("item_id == 211301")
avg_butter_prices = butter.groupby('quote_date').agg({'price': ['mean', 'median']}).reset_index()

avg_butter_prices.columns = ['date', 'Mean', 'Median']
avg_butter_prices

,date,Mean,Median
0,198802.0,0.518299,0.510
1,198803.0,0.525400,0.520
2,198804.0,0.527150,0.515
3,198805.0,0.526853,0.510
4,198806.0,0.530350,0.510
...,...,...,...
293,201209.0,1.362917,1.400
294,201210.0,1.345252,1.400
295,201211.0,1.343264,1.390
296,201212.0,1.398298,1.450


To use it in Altair/Vega-lite, we just have to melt it from wide to long.

In [17]:
avg_butter_prices_melted = avg_butter_prices.melt(id_vars=['date'],
                                                    var_name='price_type',
                                                    value_name='price')

# Format the date column for Altair
avg_butter_prices_melted['date'] = pd.to_datetime(avg_butter_prices_melted['date'], format='%Y%m')

avg_butter_prices_melted

,date,price_type,price
0,1988-02-01,Mean,0.518299
1,1988-03-01,Mean,0.525400
2,1988-04-01,Mean,0.527150
3,1988-05-01,Mean,0.526853
4,1988-06-01,Mean,0.530350
...,...,...,...
591,2012-09-01,Median,1.400000
592,2012-10-01,Median,1.400000
593,2012-11-01,Median,1.390000
594,2012-12-01,Median,1.450000


In [21]:
alt.Chart(avg_butter_prices_melted).mark_line().encode(
    x=alt.X('date:T', title=''),
    y=alt.Y('price:Q', title='Price (GBP)'),
    color='price_type:N'
).properties(
    title={
        "text": "Butter Prices",
        "subtitle": ["Mean and Median Prices for Butter)", "Source: ONS microdata via Davies (2021)"],
        "fontSize": 16
    }
)

alt.Chart(...)